# Case 2: Caspase Inactive & Viability Active

Analysis of compounds where:
- **Caspase = Inactive** (has at least one Inactive outcome AND no Active outcomes)
- **Viability = Active** (has at least one Active outcome)

**Classification Method:** STRICT
- Active takes precedence: if ANY test shows Active, the CID is classified as Active
- Inactive requires explicit Inactive outcome (not just absence of Active)
- CIDs with only Inconclusive outcomes are excluded

**Biological Interpretation:** Non-apoptotic cell death (necrosis)

**Assay Types:**
- `caspase` - Caspase-3/7 assay (14 cell lines)
- `viability` - Viability assay (CHO-K1)
- `mitochondrial_viability_hepg2` - Mitochondrial viability (HepG2)
- `caspase_viability_hepg2` - Caspase viability counter-screen (HepG2)

In [21]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 100)

# Load the merged dataset
df = pd.read_csv('output/merged_caspase_viability_final.csv')

print(f"Dataset Shape: {df.shape}")
print(f"Unique CIDs: {df['CID'].nunique():,}")
print(f"\nAssay Types: {df['assay_type'].unique().tolist()}")
print(f"\nCell Lines ({df['cell_line'].nunique()}): {df['cell_line'].unique().tolist()}")

Dataset Shape: (26939, 12)
Unique CIDs: 8,777

Assay Types: ['viability', 'caspase', 'mitochondrial_viability_hepg2', 'caspase_viability_hepg2']

Cell Lines (14): ['CHO-K1', 'HepG2', 'SHSY5Y', 'Hek293', 'HUV-EC-C', 'Jurkat', 'H-4-II-E', 'N2a', 'Mesangial', 'NIH 3T3', 'SK-N-SH', 'BJ', 'MRC-5', 'Renal_Proximal_Tubule']


In [22]:
# Define viability types
VIABILITY_TYPES = ['viability', 'mitochondrial_viability_hepg2', 'caspase_viability_hepg2']

print("Viability assay types used in this analysis:")
print(f"  - viability (CHO-K1)")
print(f"  - mitochondrial_viability_hepg2 (HepG2)")
print(f"  - caspase_viability_hepg2 (HepG2)")

Viability assay types used in this analysis:
  - viability (CHO-K1)
  - mitochondrial_viability_hepg2 (HepG2)
  - caspase_viability_hepg2 (HepG2)


---
## Block 1: Overview - Case 2 Criteria (STRICT Method)

Find CIDs where:
- Caspase = **Inactive** (has at least one Inactive outcome AND no Active outcomes)
- Viability = **Active** (has at least one Active outcome)

**Note:** Only CIDs tested in BOTH assay types are included.

In [30]:
# STRICT METHOD: 
# - Active takes precedence
# - Inactive = has Inactive outcome AND no Active outcome
# Case 2 = Caspase Truly Inactive AND Viability Active

# Get all CIDs tested in caspase assays
caspase_df = df[df['assay_type'] == 'caspase']
caspase_all_cids = set(caspase_df['CID'].unique())
caspase_active_cids = set(caspase_df[caspase_df['outcome'] == 'Active']['CID'].unique())
caspase_inactive_cids = set(caspase_df[caspase_df['outcome'] == 'Inactive']['CID'].unique())

# Get all CIDs tested in viability assays
viability_df = df[df['assay_type'].isin(VIABILITY_TYPES)]
viability_all_cids = set(viability_df['CID'].unique())
viability_active_cids = set(viability_df[viability_df['outcome'] == 'Active']['CID'].unique())

# Common CIDs - tested in BOTH assay types
common_cids = caspase_all_cids.intersection(viability_all_cids)

# Caspase Truly Inactive = has Inactive outcome but NO Active outcome (in common CIDs)
caspase_truly_inactive = caspase_inactive_cids.intersection(common_cids) - caspase_active_cids

# Viability Active in common CIDs
viability_active_in_common = viability_active_cids.intersection(common_cids)

print("STRICT METHOD: Active takes precedence, Inactive = has Inactive & no Active")
print("="*70)
print(f"CIDs tested in Caspase assays:     {len(caspase_all_cids):,}")
print(f"CIDs tested in Viability assays:   {len(viability_all_cids):,}")
print(f"CIDs tested in BOTH (common):      {len(common_cids):,}")
print()
print(f"Caspase Active CIDs:               {len(caspase_active_cids):,}")
print(f"Caspase Inactive CIDs:             {len(caspase_inactive_cids):,}")
print(f"Caspase Truly Inactive (in common): {len(caspase_truly_inactive):,}")
print(f"Viability Active CIDs (in common): {len(viability_active_in_common):,}")
print()

# Case 2: Caspase Truly Inactive AND Viability Active
case2_cids = caspase_truly_inactive.intersection(viability_active_in_common)

print(f"CASE 2: Caspase=Inactive AND Viability=Active: {len(case2_cids):,}")
print()
print("These compounds cause cytotoxicity via NON-apoptotic mechanisms ")

STRICT METHOD: Active takes precedence, Inactive = has Inactive & no Active
CIDs tested in Caspase assays:     7,716
CIDs tested in Viability assays:   8,734
CIDs tested in BOTH (common):      7,673

Caspase Active CIDs:               324
Caspase Inactive CIDs:             7,534
Caspase Truly Inactive (in common): 7,296
Viability Active CIDs (in common): 1,252

CASE 2: Caspase=Inactive AND Viability=Active: 943

These compounds cause cytotoxicity via NON-apoptotic mechanisms 


In [31]:
# Show these CIDs
print(f"List of {len(case2_cids):,} CIDs with Caspase=Never Active AND Viability=Active:")
print(sorted(case2_cids)[:100])  # Show first 100
if len(case2_cids) > 100:
    print(f"... and {len(case2_cids) - 100} more")

List of 943 CIDs with Caspase=Never Active AND Viability=Active:
[6, 177, 289, 370, 403, 671, 785, 798, 807, 980, 1091, 1123, 1176, 1493, 1730, 1807, 1967, 1982, 1988, 2078, 2194, 2265, 2361, 2374, 2375, 2378, 2406, 2449, 2450, 2453, 2466, 2477, 2519, 2603, 2722, 2730, 2801, 2812, 2871, 3017, 3053, 3074, 3082, 3220, 3224, 3225, 3259, 3278, 3333, 3339, 3346, 3352, 3397, 3598, 3610, 3627, 3686, 3712, 3715, 3749, 3760, 3784, 3884, 3899, 3957, 4030, 4055, 4066, 4106, 4115, 4130, 4420, 4474, 4477, 4485, 4495, 4497, 4499, 4534, 4594, 4641, 4685, 4764, 4766, 4778, 4793, 4837, 4921, 4931, 4933, 4936, 4947, 5125, 5265, 5333, 5334, 5339, 5392, 5475, 5482]
... and 843 more


---
## Block 2: Case 2 Analysis by Cell Line (SAME CELL LINE)

For each cell line, find CIDs where Caspase=Inactive AND Viability=Active in the SAME cell line.

In [25]:
# Analysis by Cell Line - Caspase Inactive & Viability Active (SAME CELL LINE)
print("="*70)
print("CASE 2: CASPASE=INACTIVE AND VIABILITY=ACTIVE BY CELL LINE")
print("="*70)
print("Note: Viability data exists only for HepG2 and CHO-K1")
print()

# Get caspase and viability dataframes
caspase_df = df[df['assay_type'] == 'caspase']
viability_df = df[df['assay_type'].isin(VIABILITY_TYPES)]

caspase_cell_lines = sorted(caspase_df['cell_line'].unique())

results = []
for cell in caspase_cell_lines:
    # Total CIDs tested for caspase in this cell line
    caspase_in_cell = caspase_df[caspase_df['cell_line'] == cell]
    total_caspase_tested = caspase_in_cell['CID'].nunique()
    
    # Caspase Inactive in this cell line
    casp_inactive = set(caspase_in_cell[caspase_in_cell['outcome'] == 'Inactive']['CID'].unique())
    
    # Viability Active in THIS SAME cell line
    viability_in_cell = viability_df[viability_df['cell_line'] == cell]
    total_viability_tested = viability_in_cell['CID'].nunique()
    viab_active = set(viability_in_cell[viability_in_cell['outcome'] == 'Active']['CID'].unique())
    
    # Case 2: Caspase Inactive AND Viability Active in SAME cell line
    case2_in_cell = casp_inactive.intersection(viab_active)
    
    results.append({
        'Cell Line': cell,
        'Caspase Tested': total_caspase_tested,
        'Caspase Inactive': len(casp_inactive),
        'Viability Tested': total_viability_tested,
        'Viability Active': len(viab_active),
        'Case2 (Both)': len(case2_in_cell),
        'Case2 Rate (%)': round(len(case2_in_cell)/len(casp_inactive)*100, 2) if len(casp_inactive) > 0 else 0
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Caspase Inactive', ascending=False)
print(results_df.to_string(index=False))

# Summary for cell lines WITH viability data
print(f"\n" + "-"*70)
hepg2_case2 = results_df[results_df['Cell Line'] == 'HepG2']['Case2 (Both)'].values[0]
chok1_case2 = results_df[results_df['Cell Line'] == 'CHO-K1']['Case2 (Both)'].values[0]
print(f"HepG2: {hepg2_case2} CIDs with Caspase=Inactive AND Viability=Active")
print(f"CHO-K1: {chok1_case2} CIDs with Caspase=Inactive AND Viability=Active")

CASE 2: CASPASE=INACTIVE AND VIABILITY=ACTIVE BY CELL LINE
Note: Viability data exists only for HepG2 and CHO-K1

            Cell Line  Caspase Tested  Caspase Inactive  Viability Tested  Viability Active  Case2 (Both)  Case2 Rate (%)
                HepG2            7671              7299              8734               754           492            6.74
               CHO-K1             394               199              7671              1155           127           63.82
             HUV-EC-C              27                22                 0                 0             0            0.00
               Jurkat             106                 4                 0                 0             0            0.00
               SHSY5Y              23                 3                 0                 0             0            0.00
                   BJ               5                 2                 0                 0             0            0.00
            Mesangial           

In [26]:
# HepG2 Specific: Case 2 IN THE SAME CELL LINE
print("="*70)
print("HepG2: CASPASE=INACTIVE AND VIABILITY=ACTIVE (SAME CELL LINE)")
print("="*70)

# Caspase Inactive in HepG2
caspase_inactive_hepg2 = set(df[
    (df['assay_type'] == 'caspase') & 
    (df['cell_line'] == 'HepG2') & 
    (df['outcome'] == 'Inactive')
]['CID'].unique())

# Viability Active in HepG2 (mitochondrial_viability_hepg2 + caspase_viability_hepg2)
viability_active_hepg2 = set(df[
    (df['assay_type'].isin(['mitochondrial_viability_hepg2', 'caspase_viability_hepg2'])) & 
    (df['cell_line'] == 'HepG2') & 
    (df['outcome'] == 'Active')
]['CID'].unique())

case2_hepg2 = caspase_inactive_hepg2.intersection(viability_active_hepg2)

print(f"\nCaspase Inactive in HepG2: {len(caspase_inactive_hepg2):,}")
print(f"Viability Active in HepG2 (mito + caspase_viability): {len(viability_active_hepg2):,}")
print(f"\n>>> CASE 2 in HepG2: {len(case2_hepg2):,}")

HepG2: CASPASE=INACTIVE AND VIABILITY=ACTIVE (SAME CELL LINE)

Caspase Inactive in HepG2: 7,299
Viability Active in HepG2 (mito + caspase_viability): 754

>>> CASE 2 in HepG2: 492


In [27]:
# CHO-K1 Specific: Case 2 IN THE SAME CELL LINE
print("="*70)
print("CHO-K1: CASPASE=INACTIVE AND VIABILITY=ACTIVE (SAME CELL LINE)")
print("="*70)

# Caspase Inactive in CHO-K1
caspase_inactive_chok1 = set(df[
    (df['assay_type'] == 'caspase') & 
    (df['cell_line'] == 'CHO-K1') & 
    (df['outcome'] == 'Inactive')
]['CID'].unique())

# Viability Active in CHO-K1
viability_active_chok1 = set(df[
    (df['assay_type'] == 'viability') & 
    (df['cell_line'] == 'CHO-K1') & 
    (df['outcome'] == 'Active')
]['CID'].unique())

case2_chok1 = caspase_inactive_chok1.intersection(viability_active_chok1)

print(f"\nCaspase Inactive in CHO-K1: {len(caspase_inactive_chok1):,}")
print(f"Viability Active in CHO-K1: {len(viability_active_chok1):,}")
print(f"\n>>> CASE 2 in CHO-K1: {len(case2_chok1):,}")

CHO-K1: CASPASE=INACTIVE AND VIABILITY=ACTIVE (SAME CELL LINE)

Caspase Inactive in CHO-K1: 199
Viability Active in CHO-K1: 1,155

>>> CASE 2 in CHO-K1: 127


---
## Block 3: Export Data - Case 2

Save the data to CSV for easy verification, grouped by CID.

In [28]:
# Case 2: Export CIDs where Caspase=Inactive (truly) AND Viability=Active
print("="*70)
print("CASE 2: EXPORTING CASPASE INACTIVE & VIABILITY ACTIVE DATA")
print("="*70)

print(f"\nTotal CIDs (strict method): {len(case2_cids):,}")

# Get records for these CIDs:
# - All Caspase records (to show outcomes - should have Inactive but no Active)
# - Viability Active records
case2_caspase = df[
    (df['CID'].isin(case2_cids)) & 
    (df['assay_type'] == 'caspase')
][['CID', 'SMILES', 'assay_type', 'cell_line', 'outcome', 'activity_score', 'AID', 'assay_name']].copy()

case2_viability = df[
    (df['CID'].isin(case2_cids)) & 
    (df['assay_type'].isin(VIABILITY_TYPES)) &
    (df['outcome'] == 'Active')
][['CID', 'SMILES', 'assay_type', 'cell_line', 'outcome', 'activity_score', 'AID', 'assay_name']].copy()

# Combine
case2_data = pd.concat([case2_caspase, case2_viability], ignore_index=True)
case2_data = case2_data.sort_values(['CID', 'cell_line', 'assay_type'])

print(f"Total records: {len(case2_data):,}")
print(f"\nPreview:")
print(case2_data.head(20).to_string(index=False))

# Verify: Caspase outcomes should have Inactive but NO Active
caspase_outcomes = case2_caspase['outcome'].value_counts()
print(f"\nCaspase outcomes for Case 2 CIDs (should have Inactive, NO Active):")
print(caspase_outcomes)

# Save to CSV
output_path = 'output/case2_caspase_inactive_viability_active.csv'
case2_data.to_csv(output_path, index=False)
print(f"\n>>> Saved to: {output_path}")
print(f"    Shape: {case2_data.shape}")

CASE 2: EXPORTING CASPASE INACTIVE & VIABILITY ACTIVE DATA

Total CIDs (strict method): 943
Total records: 2,515

Preview:
 CID                                    SMILES                    assay_type cell_line      outcome  activity_score     AID                   assay_name
   6 C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl                     viability    CHO-K1       Active              83 1346979 AID_1346979_viability_CHO-K1
   6 C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl                       caspase     HepG2     Inactive               0 1346978  AID_1346978_caspase37_HepG2
   6 C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl                       caspase     HepG2 Inconclusive              10 1346978  AID_1346978_caspase37_HepG2
   6 C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl mitochondrial_viability_hepg2     HepG2       Active              81  720634        AID_720634_mito_HepG2
 177                                      CC=O                     viability    CHO-K1       Active             

In [29]:
# Create summary table grouped by CID for easy count verification
print("="*70)
print("CASE 2: SUMMARY BY CID (STRICT METHOD)")
print("="*70)

# For each CID, get the caspase outcomes and viability active cell lines
caspase_by_cid = case2_caspase.groupby('CID').agg({
    'cell_line': lambda x: ', '.join(sorted(x.unique())),
    'outcome': lambda x: ', '.join(sorted(x.unique()))
}).reset_index()
caspase_by_cid.columns = ['CID', 'Caspase_CellLines', 'Caspase_Outcomes']

viability_active_by_cid = case2_viability.groupby('CID').agg({
    'cell_line': lambda x: ', '.join(sorted(x.unique())),
    'assay_type': lambda x: ', '.join(sorted(x.unique())),
    'activity_score': 'max'
}).reset_index()
viability_active_by_cid.columns = ['CID', 'Viability_Active_CellLines', 'Viability_AssayTypes', 'Max_Viability_Score']

# Merge
case2_summary = pd.merge(caspase_by_cid, viability_active_by_cid, on='CID')

# Add SMILES
smiles_map = df[['CID', 'SMILES']].drop_duplicates().set_index('CID')['SMILES'].to_dict()
case2_summary['SMILES'] = case2_summary['CID'].map(smiles_map)

case2_summary = case2_summary.sort_values('CID')

print(f"\nTotal unique CIDs: {len(case2_summary):,}")
print(f"\nSummary table (first 30 rows):")
print(case2_summary.head(30).to_string(index=False))

# Save summary to CSV
summary_path = 'output/case2_summary_by_cid.csv'
case2_summary.to_csv(summary_path, index=False)
print(f"\n>>> Summary saved to: {summary_path}")
print(f"    Total CIDs: {len(case2_summary):,}")

CASE 2: SUMMARY BY CID (STRICT METHOD)

Total unique CIDs: 943

Summary table (first 30 rows):
 CID Caspase_CellLines       Caspase_Outcomes Viability_Active_CellLines                     Viability_AssayTypes  Max_Viability_Score                                                          SMILES
   6             HepG2 Inactive, Inconclusive              CHO-K1, HepG2 mitochondrial_viability_hepg2, viability                   83                       C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl
 177             HepG2 Inactive, Inconclusive                     CHO-K1                                viability                   84                                                            CC=O
 289     HepG2, SHSY5Y Inactive, Inconclusive                     CHO-K1                                viability                   40                                               C1=CC=C(C(=C1)O)O
 370             HepG2               Inactive              CHO-K1, HepG2       caspase_viability_hepg2, viabi